# Domain Adaptation

## Learning Objectives
1. Visualise domain shift in 2D and measure Maximum Mean Discrepancy (MMD).
2. Implement CORAL alignment in PyTorch to minimise covariance shift.
3. Fine-tune a source-trained model on small labelled target data.
4. Apply pseudo-label self-training and DANN adversarial adaptation.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Domain Shift Visualisation and MMD (NumPy)

In [ ]:
# Simulate domain shift: source and target have same classes but different distributions.

def make_domain(n, shift, scale=1.0, seed=0):
    rng = np.random.default_rng(seed)
    X0 = rng.standard_normal((n//2, 2)) * scale + np.array([-2, 0]) + shift
    X1 = rng.standard_normal((n//2, 2)) * scale + np.array([ 2, 0]) + shift
    return np.vstack([X0, X1]), np.array([0]*(n//2) + [1]*(n//2))

X_src, y_src = make_domain(200, shift=(0, 0), scale=0.8, seed=0)
X_tgt, y_tgt = make_domain(200, shift=(3, 2), scale=1.4, seed=1)

def mmd_rbf(X1, X2, sigma=1.0):
    # Maximum Mean Discrepancy with RBF kernel: 0 if same distribution.
    def k(A, B):
        dists = ((A[:, None] - B[None, :]) ** 2).sum(-1)
        return np.exp(-dists / (2 * sigma ** 2))
    return k(X1, X1).mean() - 2*k(X1, X2).mean() + k(X2, X2).mean()

mmd = mmd_rbf(X_src, X_tgt)
print(f"MMD (source vs target): {mmd:.4f}  (higher = more shift)")

from sklearn.linear_model import LogisticRegression
clf_src_base = LogisticRegression(max_iter=500).fit(X_src, y_src)
acc_src = clf_src_base.score(X_src, y_src)
acc_tgt = clf_src_base.score(X_tgt, y_tgt)
print(f"Source accuracy: {acc_src:.4f}")
print(f"Target accuracy (no adaptation): {acc_tgt:.4f}  -- degraded by shift")
print(f"Accuracy drop due to shift: {acc_src - acc_tgt:+.4f}")


## Level 2: CORAL Alignment in PyTorch

In [ ]:
# CORAL: align second-order statistics (covariance) of source and target features.

def coral_loss(source, target):
    # Frobenius norm of covariance matrix difference.
    ns, d = source.shape
    nt    = target.shape[0]
    Cs = (source.T @ source
          - source.sum(0).unsqueeze(1) @ source.sum(0).unsqueeze(0) / ns) / (ns - 1)
    Ct = (target.T @ target
          - target.sum(0).unsqueeze(1) @ target.sum(0).unsqueeze(0) / nt) / (nt - 1)
    return (Cs - Ct).pow(2).sum() / (4 * d * d)

class CoralEncoder(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.encoder    = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(),
                                        nn.Linear(d_hidden, d_hidden), nn.ReLU())
        self.classifier = nn.Linear(d_hidden, 2)
    def forward(self, x):
        feat = self.encoder(x)
        return feat, self.classifier(feat)

X_src_t = torch.tensor(X_src, dtype=torch.float32, device=device)
y_src_t = torch.tensor(y_src, dtype=torch.long,    device=device)
X_tgt_t = torch.tensor(X_tgt, dtype=torch.float32, device=device)
y_tgt_t = torch.tensor(y_tgt, dtype=torch.long,    device=device)

model_coral = CoralEncoder(2, 32).to(device)
opt_coral   = torch.optim.Adam(model_coral.parameters(), lr=1e-3)
LAMBDA_CORAL = 0.5

for epoch in range(150):
    model_coral.train()
    opt_coral.zero_grad()
    try:
        feat_s, logits_s = model_coral(X_src_t)
        feat_t, _        = model_coral(X_tgt_t)
        cls_loss   = nn.CrossEntropyLoss()(logits_s, y_src_t)
        align_loss = coral_loss(feat_s, feat_t)
        (cls_loss + LAMBDA_CORAL * align_loss).backward()
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print("OOM -- reduce hidden size")
            torch.cuda.empty_cache()
            continue
        raise
    opt_coral.step()

model_coral.eval()
with torch.no_grad():
    acc_coral = (model_coral(X_tgt_t)[1].argmax(1) == y_tgt_t).float().mean().item()
print(f"CORAL target accuracy: {acc_coral:.4f}  (baseline: {acc_tgt:.4f})")


## Real-World Example 1: Fine-Tuning for Domain Shift

In [ ]:
# Fine-tuning: the most practical approach when small labelled target data is available.

def build_clf_net(d_in, d_hidden, n_cls):
    return nn.Sequential(
        nn.Linear(d_in, d_hidden), nn.ReLU(),
        nn.Linear(d_hidden, d_hidden), nn.ReLU(),
        nn.Linear(d_hidden, n_cls)
    ).to(device)

src_model = build_clf_net(2, 64, 2)
opt_src = torch.optim.Adam(src_model.parameters(), lr=1e-3)
for _ in range(100):
    opt_src.zero_grad()
    nn.CrossEntropyLoss()(src_model(X_src_t), y_src_t).backward()
    opt_src.step()

N_FT = 20   # only 20 labelled target samples
ft_model = build_clf_net(2, 64, 2)
ft_model.load_state_dict(src_model.state_dict())   # start from source weights
opt_ft = torch.optim.Adam(ft_model.parameters(), lr=5e-4)   # lower LR for fine-tuning

for _ in range(80):
    opt_ft.zero_grad()
    idx = torch.randperm(len(X_tgt_t))[:N_FT]
    nn.CrossEntropyLoss()(ft_model(X_tgt_t[idx]), y_tgt_t[idx]).backward()
    opt_ft.step()

src_model.eval(); ft_model.eval()
with torch.no_grad():
    acc_src_on_tgt = (src_model(X_tgt_t).argmax(1) == y_tgt_t).float().mean().item()
    acc_ft_on_tgt  = (ft_model(X_tgt_t).argmax(1) == y_tgt_t).float().mean().item()
print(f"Source-only on target:          {acc_src_on_tgt:.4f}")
print(f"Fine-tuned ({N_FT} target samples): {acc_ft_on_tgt:.4f}")
print(f"Improvement: {acc_ft_on_tgt - acc_src_on_tgt:+.4f}")


## Real-World Example 2: Pseudo-Label Self-Training

In [ ]:
# Self-training: use high-confidence source-model predictions on unlabelled
# target data as pseudo-labels, then retrain on source + pseudo-labelled target.

def self_training_loop(src_model, X_tgt, y_tgt_true, n_rounds=3, conf_thr=0.80):
    model = build_clf_net(2, 64, 2)
    model.load_state_dict(src_model.state_dict())
    for rnd in range(n_rounds):
        model.eval()
        with torch.no_grad():
            probs  = F.softmax(model(X_tgt), dim=-1)
            conf, pseudo = probs.max(dim=-1)
            mask = conf >= conf_thr
        if mask.sum() > 0:
            X_comb = torch.cat([X_src_t, X_tgt[mask]], dim=0)
            y_comb = torch.cat([y_src_t, pseudo[mask]], dim=0)
        else:
            X_comb, y_comb = X_src_t, y_src_t
        model.train()
        opt_st = torch.optim.Adam(model.parameters(), lr=5e-4)
        for _ in range(50):
            idx = torch.randperm(len(X_comb))
            opt_st.zero_grad()
            nn.CrossEntropyLoss()(model(X_comb[idx]), y_comb[idx]).backward()
            opt_st.step()
        model.eval()
        with torch.no_grad():
            acc = (model(X_tgt).argmax(1) == y_tgt_true).float().mean().item()
        print(f"  Round {rnd+1}: pseudo_labels={mask.sum().item():3d}  target_acc={acc:.4f}")
    return acc

print("Pseudo-label self-training:")
final_acc = self_training_loop(src_model, X_tgt_t, y_tgt_t, n_rounds=3, conf_thr=0.80)
print(f"Final accuracy after self-training: {final_acc:.4f}")


## Real-World Example 3: DANN Adversarial Domain Adaptation

In [ ]:
# DANN: gradient reversal makes features domain-invariant while preserving class info.

class GradReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None

class DANN(nn.Module):
    def __init__(self, d_in, d_hid, n_cls):
        super().__init__()
        self.encoder    = nn.Sequential(nn.Linear(d_in, d_hid), nn.ReLU(),
                                        nn.Linear(d_hid, d_hid), nn.ReLU())
        self.label_clf  = nn.Linear(d_hid, n_cls)
        self.domain_clf = nn.Sequential(nn.Linear(d_hid, 16), nn.ReLU(), nn.Linear(16, 2))

    def forward(self, x, alpha=1.0):
        feat        = self.encoder(x)
        lbl_logits  = self.label_clf(feat)
        feat_rev    = GradReversal.apply(feat, alpha)
        dom_logits  = self.domain_clf(feat_rev)
        return lbl_logits, dom_logits

dann = DANN(2, 32, 2).to(device)
opt_dann  = torch.optim.Adam(dann.parameters(), lr=1e-3)
src_dom = torch.zeros(len(X_src_t), dtype=torch.long, device=device)
tgt_dom = torch.ones(len(X_tgt_t),  dtype=torch.long, device=device)

for epoch in range(150):
    dann.train()
    opt_dann.zero_grad()
    alpha = 2.0 / (1 + np.exp(-10 * epoch / 150)) - 1   # ramp from 0 to 1
    lbl_logits,  dom_src = dann(X_src_t, alpha)
    _,           dom_tgt = dann(X_tgt_t, alpha)
    cls_loss = nn.CrossEntropyLoss()(lbl_logits, y_src_t)
    dom_loss = (nn.CrossEntropyLoss()(dom_src, src_dom) +
                nn.CrossEntropyLoss()(dom_tgt, tgt_dom)) / 2
    (cls_loss + dom_loss).backward()
    opt_dann.step()

dann.eval()
with torch.no_grad():
    acc_dann = (dann(X_tgt_t)[0].argmax(1) == y_tgt_t).float().mean().item()
print(f"DANN   target accuracy: {acc_dann:.4f}")
print(f"CORAL  target accuracy: {acc_coral:.4f}")
print(f"Source-only baseline:   {acc_src_on_tgt:.4f}")
print("DANN trains domain-invariant features via adversarial gradient reversal.")
